<a href="https://colab.research.google.com/github/Text-Machine/mask-predict/blob/main/chr-paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

# CHR Paper notebook

This notebook aims to reproduce all figures and tables that inform the analysis of the CHR paper.

In [ ]:
!git clone https://github.com/Text-Machine/mask-predict.git

In [ ]:
%cd mask-predict

In [ ]:
!pip install -q -e .

In [ ]:
!gdown 125wfZ1P9MfFZ19XS2SCfallsnoNhLCUf

In [ ]:
!unzip -o "/content/chr-data.zip"

In [ ]:
import pandas as pd
import json
from tqdm import tqdm
from explain import *
from pathlib import Path
from collections import Counter
import seaborn as sns

In [ ]:
collection,genre_suffix = 'blb',''
if collection == 'blb':
  genre_suffix = '_with_genre'

TargetMaskedToken = 'machine' # the token to be masked in the target sentence

originalFolder = 'masking_data' # change to '.' when working in colab
dataPath = 'input_data' # change to '.' when working in colab 
processedFolder = 'gradient_data' # change '.' when working in colab
predCol = "pred_bert_1760_1900"
resultType = 'pred_kw_filtered' # pred | pred_kw_filter

print(f"This analysis focuses on '{TargetMaskedToken}'.")

We are loading the data frame with deduplicated sentences.

In [ ]:
df_sent_all = pd.read_csv(f'{originalFolder}/{collection}_{TargetMaskedToken}_clusters{genre_suffix}_deduplicated.tsv', index_col=0, sep='\t').reset_index(drop=True)
print(f'We have {df_sent_all.shape[0]} unique sentences for the target token {TargetMaskedToken} in the {collection} collection.')


In [ ]:
# load the original sentences with the predicted tokens
df_sent = pd.read_csv(f'{dataPath}/{collection}_{TargetMaskedToken}{genre_suffix}_{resultType}.tsv', index_col=0, sep='\t').reset_index(drop=True)
print(f'We have {df_sent.shape[0]} sentences that produced human predictions for the target token {TargetMaskedToken} in the {collection} collection.')
df_ig = pd.read_csv(f'{processedFolder}/results_{collection}_{TargetMaskedToken}_{resultType}_processed.csv', index_col=0 )
print(f'We have {df_ig.shape[0]} explanations for the target token {TargetMaskedToken} in the {collection} collection.')


Here we load the words that we selected as human prediction, i.e. these are words predicted by BLERT possible referring to human fillers for the masked machine token.

In [ ]:
with open(f'{dataPath}/250_freq_pred_KB_edit.txt') as f:
    human_words = f.read().splitlines()

print(human_words[:10])

We create a new column where filter the prediction, only retaining the human words.

In [ ]:
for colName in ['pred_bert_contemporary', 'pred_bert_1760_1900']:
    df_sent[f'{colName}_human'] = df_sent[colName].apply(
        lambda x: {w:s for w, s in dict(eval(x)).items() if w in human_words})


We look at results by decade, therefore adding decade column to the data frames

In [ ]:
df_sent_all['decade'] = df_sent_all['date'].apply(lambda x: int(x/10)*10)
df_sent['decade'] = df_sent['date'].apply(lambda x: int(x/10)*10)

### Analysis: Distribution of predictions

BLERT is not always equally convinced of it's predictions. Below we plot the distribution of it confidence scores for the human words it predicted instead of machine. These are the probabilities that a word fits the given context in the position of the masked token. We see that 

In [ ]:
import itertools
scores_human = list(itertools.chain.from_iterable(df_sent['pred_bert_1760_1900_human'].apply(lambda x: list(x.values()))))
scores_all = list(itertools.chain.from_iterable(df_sent['pred_bert_1760_1900'].apply(lambda x: list(dict(eval(x)).values()))))

print("Distribution of the human prediction scores for the target token '{TargetMaskedToken}' in the {collection} collection.")
#pd.Series(scores_human).plot(kind='density')
pd.Series(scores_human).hist(bins=100, alpha=0.9, label='human predictions')
#pd.Series(scores_all).plot(kind='density', alpha=0.5)


### Analysis: timeline for all predictions

Below we plot the number of unique sentences containing 'human' predictions for masked machine tokens. We plot by decade, for all the human predictions, higher than a set confidence threshold. 

While maybe not drastically, we observe a small rise in the relative number of 'atypical' sentences or 'atypical' language use.

In [ ]:
time_unit = 'decade' # change to 'date' for yearly analysis
thresholds = [0.1, 0.2, 0.5]
for threshold in thresholds:
    df_sent_filtered = df_sent[df_sent['pred_bert_1760_1900_human'].apply(lambda x: max(list(x.values())+[.0]) > threshold)]
    (df_sent_filtered.groupby(time_unit).size()/df_sent_all.groupby(time_unit).size()).loc[1800:1899].plot(alpha=0.9,
                                                                                                            title=f'', 
                                                                                                            xlabel=f'{time_unit}',
                                                                                                            figsize=(15, 4))


### Analysis: timeline for a selected thema

By changing the `wordList` variable below, we can plot timelines for specific subset or theme of predictions. 

The children theme shows a pronounced upward trend. This could be one of the subquestions we address in the paper: how and why the increasing confusion of child and machine?

In [ ]:
time_unit = 'decade' # change to 'date' for yearly analysis
thresholds = [ 0.1, 0.2, 0.5] # compare the results for different thresholds to see how the results change based on the confidence in the predictions
wordList = ['child','children', 'boy','boys','girl','girls']  # change these words to analyze a different theme
for threshold in thresholds:
    df_sent_filtered = df_sent[df_sent[f'{colName}_human'].apply(
        lambda x: max(list({w:s for w,s in x.items() if w in wordList}.values())+[.0]) > threshold
                )
            ]
    (df_sent_filtered.groupby(time_unit).size() / df_sent_all.groupby(time_unit).size()).loc[1800:1899].plot(alpha=0.8, 
                                                                                                             title=f'',
                                                                                                               xlabel=f'{time_unit}',
                                                                                                               figsize=(15, 4))


### Analysis: results by genre

The results hold when splitting the data by genre. Upward trend appears in both fiction and non-fiction but, more articulate in the form}er.

In [ ]:

time_unit = 'decade' # change to 'date' for yearly analysis
threshold = 0.1 # use higher thresholds for more conservative analysis, i.e. only sentences with a high probability of the tokens in wordList being predicted are included in the analysis
wordList = ['child','children', 'boy','boys','girl','girls'] # change these words to analyze a different theme

df_sent_filtered = df_sent[df_sent['pred_bert_1760_1900_human'].apply(lambda x: max(list({w:s for w,s in x.items() if w in wordList}.values())+[.0]) > threshold)]
plot_df = df_sent_filtered.groupby([time_unit, 'genre']).size() / df_sent_all.groupby([time_unit, 'genre']).size()
plot_df_rel = (df_sent_filtered.groupby([time_unit, 'genre']).size() / df_sent_all.groupby([time_unit, 'genre']).size()).reset_index()
sns.lineplot(data=plot_df_rel, x=time_unit, y=0, hue='genre', marker='o')

### Analysis: Distribution of prediction

Which human words does BLERT predict instead of ''machine''? And why? 

We refined the analysis below by allowing to refine the threshold, i.e. to focus only on "confident" predictions etc.

In [ ]:

thresholds = 0.1 # change this threshold to see how the results change based on the confidence in the predictions


In [ ]:
df_sent_filtered = df_sent[df_sent[f'{colName}_human'].apply(lambda x: max(list(x.values())+[.0]) > threshold)]
preds = Counter([w for l in df_sent_filtered[f'{colName}'].values for w,s in eval(l) if (w in human_words) & (s > threshold) ])

In [ ]:
pd.DataFrame([(w, c/len(df_sent_filtered)) for w, c in preds.most_common(100)]
             ).rename(columns={0: 'Predicted token', 1: 'Count'}
                      ).set_index('Predicted token'
                                  ).plot(kind='bar', title=f'Top 100 predicted tokens for the target token {TargetMaskedToken}.', 
                                         ylabel='Count', xlabel='Predicted token', figsize=(15,5))

### Analysis: explainability

Which words drive the predictions, and especially, how to interpet this increasing "confusion" or children with machines? What does it tell us about changes in the discourse about machines.

What we compute is not just how words contribute to the prediction, this tells us often more about what type of sentences are included in our data. We look at which words explain the difference between machine and human words, i.e. which words drive the prediction towards 'human' and away from 'machine'.

More technically, for each word in the sentence we compute pairwise differences between ''machine'' and ''human'' predictions (for the masked token) for each token in the sentence.

In [ ]:
# Create row order within each id and Target
df_ig["row_idx"] = df_ig.groupby(["id", "Target"]).cumcount()

# Extract machine/machines scores
machine_scores = (
    df_ig[df_ig["Target"].isin(["machine", "machines"])]
    [["id", "row_idx", "Score_normalized"]]
    .rename(columns={"Score_normalized": "machine_score"})
)

# Match each row with the corresponding machine row
df_ig = df_ig.merge(
    machine_scores,
    on=["id", "row_idx"],
    how="left"
)

# Subtract
df_ig["diff"] = df_ig["Score_normalized"] - df_ig["machine_score"]

# Optional: remove helper column
df_ig.drop(columns=["row_idx", "machine_score"], inplace=True)

In [ ]:
df_ig[(df_ig['id'] == 11)].Target.unique()

In [ ]:
df_ig[(df_ig['id'] == 11) & (df_ig['Target'].isin(['machine', 'machines']))][['Target', 'Score_normalized', 'diff']].head(3)

In [ ]:
df_ig[(df_ig['id'] == 11) & (df_ig['Target'].isin(['girl']))][['Target', 'Score_normalized', 'diff']].head(3)

## Analysis: Explanation scores across all human word predictions

Below is a general analysis, highlighting which context words explain the drifting aways of explanations from the machine.

The threshold, again, regulates, the confidence.

The `df_result` dataframe shows the words that exhibit the highest difference between machine and human words. Below we provide tools for analysing these words, and the sentences, in which they appear, in more depth.

In [ ]:
threshold = 0.25


targetTokens = ['machine','machines'] # we look at the predictions for all the non machine words
df_comparisonConcept = df_ig[
    (~df_ig['Target'].isin(targetTokens)) & # we exclude the target token itself, as we are interested in other tokens that are predictive of the contrastive concept
    (df_ig['Target_score'].between(threshold, 1.0))
                ].groupby('Token').agg(
                        count=('id', 'count'),identifiers=('id', set),avg_diff=('diff', 'mean'), avg_score=('Score_normalized', 'mean')
                    ).reset_index()


In [ ]:
# please note that this repeats sentences, this acros all sentences with all the filtered keywords
min_count = 20
df_result = df_comparisonConcept[df_comparisonConcept['count'] >= min_count].sort_values(by='avg_diff', ascending=False)
df_result.head(10)

## Analysis: Explanation scores for a specific theme


The code below, repeats the contextual analyis, but focussing on specific theme, defined by the `comparisonTokens` variable. Here we highlight which context words explain the drifting aways of explanations from the machine, with regard to a specific theme.

The `threshold`  regulates, the confidence.

`comarisonTokens` defines a set of words which we compare against machine predictions.

The `df_result` dataframe shows the words that exhibit the highest difference between machine and human words. Below we provide tools for analysing these words, and the sentences, in which they appear, in more depth.

In [ ]:


comparisonTokens = ['child','children', 'boy','boys','girl','girls']

df_comparisonConcept = df_ig[
    (df_ig['Target'].isin(comparisonTokens)) & \
    (df_ig['Target_score'].between(0.1, 1.0)) # we exclude the target token itself, as we are interested in other tokens that are predictive of the contrastive concept
                ].groupby('Token').agg(
                        count=('id', 'count'),identifiers=('id', set),avg_diff=('diff', 'mean'), avg_score=('Score_normalized', 'mean')
                    ).reset_index()


In [ ]:
# please note that this repeats sentences, this acros all sentences with all the filtered keywords
min_count = 20
df_result = df_comparisonConcept[df_comparisonConcept['count'] >= min_count].sort_values(by='avg_diff', ascending=False)
df_result.head(10)

In [ ]:
#df_result.avg_score.plot(kind='hist', bins=100, title=f'Histogram of average scores for tokens predictive of the contrastive concept.', xlabel='Average score', ylabel='Count')

# Analysis: Zooming in on sentences based on context words

You can zoom in a sentences with using the predictive context words. Change the `id` variable below with one of the index numbers of the`df_result` dataframe.

The sentence are ranked, with examples where the context word obtains the highest scores, first.

You can change the following variable:

- `id` selected from the `df_result` dataframe
- `sort_value` how do sort the sentences, i.e. where the impact of the target word is thelargest ('Score_normalized') or makes the biggest difference ('diff')

In [ ]:
pd.set_option('display.max_colwidth', 200)

In [ ]:
id = 1637
sort_value = 'diff' #'Score_normalized' | 'diff'


identifiers_ranked = df_ig.loc[df_ig['id'].isin(list(df_result.loc[id].identifiers)) \
                             & (df_ig['Token']==df_result.loc[id].Token) \
                             & df_ig['Target'].isin(comparisonTokens)
                             
                             ].sort_values(by=sort_value, ascending=False)['id'].unique()
sentences_ranked = df_sent.iloc[list(identifiers_ranked)].head(20)
sentences_ranked.currentSentence

In [ ]:
# Optional: save and expeort sentences
name = 'child'
outPath = Path('sentences')
outPath.mkdir(exist_ok=True)
sentences_ranked.to_csv(outPath / f'{name}_sentences.csv', index=False)

In [ ]:
modelName = "Livingwithmachines/bert_1760_1900"
explainer = MaskedLMExplainer(model_name=modelName, device=pick_device())

## Analysis: Inspect IG for one sentence

In [ ]:
idx = 11509
target_token = 'predicted' # 'actual' | 'predicted'

sentence = df_sent.iloc[idx].maskedSentence

if target_token == 'actual':
    target_token = df_sent.iloc[idx].targetExpression
elif target_token == 'predicted':
    target_token =  [w for w, v in sorted(
       df_sent.iloc[idx].pred_bert_contemporary_human.items(), key=lambda x: x[1], reverse=True)
                     if w in wordList][0]
print(target_token)

In [ ]:

highlight_context_tokens(explainer, sentence, target=target_token, word_agg="mean")

## Inspect sentences based on Context and Predicted token

In [ ]:
predictedToken = 'men'
contextToken = 'animal'
sort_value = 'diff' #'Score_normalized' | 'diff'

ids = df_ig[(df_ig['Token'].str.lower() == contextToken) & (df_ig['Target'].str.lower() == predictedToken)
                    ].sort_values(sort_value,ascending=False).id.values

In [ ]:
ids

In [ ]:
idx = 4495
wordList = human_words
sentence = df_sent.iloc[idx].maskedSentence
targets = list(set(df_ig[df_ig['id'] == idx].Target.unique()).intersection(set(wordList)))
target_expression = df_sent.iloc[idx].targetExpression

In [ ]:
print(f"Sentence: {sentence}")
print(f"Targets: {targets}")
print(f"Targets: {target_expression}")

In [ ]:
#target =targets[0]

In [ ]:

highlight_context_tokens(explainer, sentence, target=predictedToken, word_agg="mean")

## Historical Experiments

In [ ]:
df_ig_merged = df_ig.merge(df_sent[['decade','date']], left_on='id',right_index=True, how='left')
df_ig.shape, df_ig_merged.shape

In [ ]:
df_ig_merged[(~df_ig_merged['Target'].isin(['machine','machines'])) & \
             (df_ig_merged['Target_score'].between(.1,1.0)) & \
             (df_ig_merged['Sentence_length'] > 20)
             ].groupby(['decade','id'])['diff'].mean().groupby('decade').mean().loc[1800:1890].plot(title=f'Average difference in scores for tokens predictive of the contrastive concept over time.', xlabel='Decade', ylabel='Average difference in scores', figsize=(15, 4))

In [ ]:
df_ig_merged.columns

In [ ]:
df_ig_merged.groupby(['decade'])['Sentence_length'].mean().loc[1800:1890].plot(title=f'Average sentence length over time.', xlabel='Decade', ylabel='Average sentence length', figsize=(15, 4))

In [ ]:
df_ig_merged[(df_ig_merged['Target'].isin(comparisonTokens)) & \
             (df_ig_merged['Target_score'].between(.1,1.0)) & \
             (df_ig_merged['Sentence_length'].between(20, 200))
             ].groupby(['decade','id'])['diff'].mean().groupby('decade').mean().loc[1800:1890].plot(title=f'Average difference in scores for tokens predictive of the contrastive concept over time.', xlabel='Decade', ylabel='Average difference in scores', figsize=(15, 4))

In [ ]:
df_ig['sent_diff'] = df_ig.groupby('id')['diff'].transform(lambda x: x.mean())
df_ig['sent_diff'] = df_ig.groupby('id')['diff'].transform(lambda x: x.mean())
sent_diff_scores = df_ig[(~df_ig['Target'].isin(['machine','machines'])) & \
                          (df_ig['Target_score'].between(.1,1.0)) & \
                          (df_ig['Sentence_length'] > 20)
                          ].drop_duplicates(subset=['id'])['sent_diff']
sent_diff_scores.plot(kind='hist', bins=100, title=f'Histogram of average difference in scores for tokens predictive of the contrastive concept.', xlabel='Average difference in scores', ylabel='Count')

In [ ]:
sent_diff_scores = df_ig[~df_ig['Target'].isin(['machine','machines'])].drop_duplicates(subset=['id'])['sent_diff']
sent_diff_scores.plot(kind='hist', bins=100, title=f'Histogram of average difference in scores for tokens predictive of the contrastive concept.', xlabel='Average difference in scores', ylabel='Count')

In [ ]:
most_changes =df_ig[(~df_ig['Target'].isin(['machine','machines'])) & \
                      (df_ig['Target_score'].between(.1,1.0)) & \
                      (df_ig['Sentence_length'] > 50)
                      ].sort_values(by='sent_diff', ascending=False)[['id','sent_diff','Target','Token']]['id'].unique()[:10]

In [ ]:
pd.set_option('display.max_colwidth', 200)
df_sent.iloc[most_changes].currentSentence

In [ ]:
most_changes =df_ig[(df_ig['Target'].isin(comparisonTokens)) & \
                      (df_ig['Target_score'].between(.1,1.0)) & \
                      (df_ig['Sentence_length'] > 100)
                      ].sort_values(by='sent_diff', ascending=False)[['id','sent_diff','Target','Token']]['id'].unique()[:10]

In [ ]:
pd.set_option('display.max_colwidth', 200)
df_sent.iloc[most_changes].currentSentence

In [ ]:
df_ig_by_token = df_ig_merged.groupby(['Token','decade'])['Score_normalized'].mean().reset_index()

In [ ]:
df_ig_by_token

In [ ]:
words = [w for w,v in Counter(df_ig['Token']).items() if v > 100]

In [ ]:
from sklearn.linear_model import LinearRegression
def get_linear_regression_slope(df, x_col='decade', y_col='Score_normalized'):
    X = df[[x_col]].values
    y = df[y_col].values
    model = LinearRegression()
    model.fit(X, y)
    return model.coef_[0] # Return the slope

In [ ]:
slopes = df_ig_by_token.groupby('Token').apply(lambda x: get_linear_regression_slope(x))
slopes.loc[words].sort_values(ascending=False)

In [ ]:
token = 'sewing'
df_ig_merged[df_ig_merged.Token==token].groupby('decade')['Score_normalized'].mean().plot(title=f'Average score for the token "{token}" over time.', xlabel='Decade', ylabel='Average score', figsize=(15, 4))